In [27]:
### PS5 (c) — Cell 1: imports and workbook inspection
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

file = "Problem_Set5_2026.xlsx"

print("Sheets:", pd.ExcelFile(file).sheet_names)

factors_raw = pd.read_excel(file, sheet_name="Fama-French factors", header=None)
data = pd.read_excel(file, sheet_name="25_Size_BEME_Portfolios", header=None)

# Factor sheet: rows 0-2 are titles/headers, monthly data start in row 3.
print("\nFama-French factors sheet, shape:", factors_raw.shape)
print(factors_raw.iloc[:5].to_string())
print("...")
print(factors_raw.tail(2).to_string())

# 25 Size x BE/ME sheet: three side-by-side blocks separated by blank columns.
print("\n25_Size_BEME_Portfolios sheet, shape:", data.shape)
print("Block titles (row 0):", data.iloc[0, [1, 28, 55]].tolist())
blocks = {
    "Returns (cols 0-25)": [0, 1, 2, 24, 25],
    "Size (cols 27-52)": [27, 28, 29, 51, 52],
    "BE/ME (cols 54-79)": [54, 55, 56, 78, 79],
}
for label, cols in blocks.items():
    print(f"\n{label}:")
    print(data.iloc[:5, cols].to_string())

print("\nSeparator columns 26 and 53 empty:",
      data[26].isna().all(), data[53].isna().all())
print("Return dates:", data.iloc[3, 0], "to", data.iloc[-1, 0])
print("Size dates:  ", data.iloc[3, 27], "to", data.iloc[-1, 27])
bm_years = data.iloc[3:, 54].dropna()
print("BE/ME years: ", int(bm_years.iloc[0]), "to", int(bm_years.iloc[-1]),
      f"({len(bm_years)} annual rows)")

Sheets: ['Fama-French factors', '25_Size_BEME_Portfolios', '25_Size_212_Portfolios']

Fama-French factors sheet, shape: (1203, 7)
                     0       1     2     3     4     5                                                    6
0  Fama-French factors     NaN   NaN   NaN   NaN   NaN                                                  NaN
1                  NaN     NaN   NaN   NaN   NaN   NaN                                                 NBER
2                  NaN  Mkt-RF   SMB   HML    RF   UMD  Recession indicator (1=recession, 0=non-recesstion)
3               192607    2.89 -2.55 -2.39  0.22  -999                                                    0
4               192608    2.64 -1.14  3.81  0.25  -999                                                    0
...
           0     1     2     3     4     5  6
1201  202605  4.91 -1.68 -2.16  0.31  1.38  0
1202  202606 -1.07  3.58  3.34  0.29  4.76  0

25_Size_BEME_Portfolios sheet, shape: (1203, 80)
Block titles (row 0): ['Averag

In [28]:
### PS5 (c) — Cell 2: load and clean PS5 data
names = [f"Size{s}_BM{b}" for s in range(1, 6) for b in range(1, 6)]

# Header labels must match across the return, size, and BE/ME blocks.
assert np.array_equal(data.iloc[1:3, 1:26], data.iloc[1:3, 28:53])
assert np.array_equal(data.iloc[1:3, 1:26], data.iloc[1:3, 55:80])

# Total portfolio returns, in percent per month (same units as source).
ret = data.iloc[3:, 1:26].astype(float)
ret.columns = names
ret.index = pd.PeriodIndex(
    data.iloc[3:, 0].astype(int).astype(str), freq="M"
)
ret = ret.mask(ret <= -99)

# Monthly average size of each portfolio.
size = data.iloc[3:, 28:53].astype(float)
size.columns = names
size.index = pd.PeriodIndex(
    data.iloc[3:, 27].astype(int).astype(str), freq="M"
)
size = size.mask(size <= 0)

# Annual average BE/ME of each portfolio, indexed by the spreadsheet's year label.
annual = data.iloc[3:, 54:80].dropna(subset=[54]).astype(float)
bm = annual.iloc[:, 1:].copy()
bm.columns = names
bm.index = annual.iloc[:, 0].astype(int)
bm = bm.mask(bm <= 0)

# Fama-French factors and RF, in percent per month (UMD/NBER not needed here).
factors = factors_raw.iloc[3:, :5].astype(float)
factors.columns = ["date", "MktRF", "SMB", "HML", "RF"]
factors.index = pd.PeriodIndex(
    factors["date"].astype(int).astype(str), freq="M"
)
factors = factors[["MktRF", "SMB", "HML", "RF"]]
factors = factors.mask(factors <= -99)

# Basic checks.
assert ret.shape[1] == 25 and size.shape[1] == 25 and bm.shape[1] == 25
assert ret.index.is_unique and size.index.is_unique
assert bm.index.is_unique and factors.index.is_unique
assert ret.index.equals(factors.index)
assert ret.index.equals(size.index)
assert ret.index.equals(
    pd.period_range(ret.index.min(), ret.index.max(), freq="M")
)
assert ret.notna().all().all() and factors.notna().all().all()
assert size.notna().all().all() and bm.notna().all().all()

# No start date is imposed: the full return/factor period is kept here and
# used for the first-pass betas. The second-pass sample is set in Cell 4.
exret = ret.sub(factors["RF"], axis=0)
rm = factors["MktRF"]
f3 = factors[["MktRF", "SMB", "HML"]]

print(f"Returns/factors/size: {exret.index[0]} to {exret.index[-1]} | "
      f"T = {len(exret)} | N = {exret.shape[1]}")
print(f"BE/ME year labels: {bm.index[0]} to {bm.index[-1]} ({len(bm)} rows)")
print("Missing values in exret / factors / size / BE/ME:",
      int(exret.isna().sum().sum()), "/", int(f3.isna().sum().sum()), "/",
      int(size.isna().sum().sum()), "/", int(bm.isna().sum().sum()))

Returns/factors/size: 1926-07 to 2026-06 | T = 1200 | N = 25
BE/ME year labels: 1926 to 2025 (100 rows)
Missing values in exret / factors / size / BE/ME: 0 / 0 / 0 / 0


In [33]:
### PS5 (c) — Cell 3: lagged characteristics and common sample (team conventions)
# ln_size: log size at t-1. ln_BM: log BE/ME of calendar year Y-1 for every month of year Y.
assert size.index.equals(pd.period_range(size.index.min(), size.index.max(), freq="M"))
ln_size_all = np.log(size).shift(1).reindex(exret.index)
ln_bm_all = np.log(bm).reindex(exret.index.year - 1)
ln_bm_all.index = exret.index

# Common sample: months where all returns, factors and characteristics exist.
valid = (exret.notna().all(axis=1) & f3.notna().all(axis=1)
         & ln_size_all.notna().all(axis=1) & ln_bm_all.notna().all(axis=1))
fm_months = exret.index[valid]
assert fm_months.equals(pd.period_range(fm_months[0], fm_months[-1], freq="M"))

exret_fm = exret.loc[fm_months]
ln_size = ln_size_all.loc[fm_months]
ln_bm = ln_bm_all.loc[fm_months]

print(f"Common sample: {fm_months[0]} to {fm_months[-1]} | T = {len(fm_months)}")
for d in [fm_months[0], fm_months[-1]]:
    print(f"Return month {d}: size from {d - 1}, BE/ME year {d.year - 1}")

Common sample: 1927-01 to 2026-06 | T = 1194
Return month 1927-01: size from 1926-12, BE/ME year 1926
Return month 2026-06: size from 2026-05, BE/ME year 2025


In [34]:
### PS5 (c) — Cell 4: fixed first-pass betas on the common sample (team conventions)
# beta_M_capm: RMRF alone (PS3 beta), used in eq (1).
# beta_M_3f, beta_SMB, beta_HML: one joint regression on RMRF, SMB, HML, used in eqs (2), (3).
betas = pd.DataFrame(index=names, dtype=float,
                     columns=["beta_M_capm", "beta_M_3f", "beta_SMB", "beta_HML"])
X_market = sm.add_constant(rm.loc[fm_months])
X_factors = sm.add_constant(f3.loc[fm_months])
for name in names:
    betas.loc[name, "beta_M_capm"] = sm.OLS(exret_fm[name], X_market).fit().params["MktRF"]
    betas.loc[name, ["beta_M_3f", "beta_SMB", "beta_HML"]] = (
        sm.OLS(exret_fm[name], X_factors).fit().params[["MktRF", "SMB", "HML"]].to_numpy())

beta_start, beta_end, beta_T = fm_months[0], fm_months[-1], len(fm_months)
assert betas.notna().all().all()
print(f"First-pass sample: {beta_start} to {beta_end} | T = {beta_T}")
print(betas.round(3).to_string())

First-pass sample: 1927-01 to 2026-06 | T = 1194
           beta_M_capm  beta_M_3f  beta_SMB  beta_HML
Size1_BM1        1.601      1.270     1.473     0.350
Size1_BM2        1.392      1.071     1.527     0.208
Size1_BM3        1.357      1.049     1.238     0.494
Size1_BM4        1.262      0.943     1.232     0.576
Size1_BM5        1.357      0.981     1.287     0.887
Size2_BM1        1.271      1.088     1.151    -0.230
Size2_BM2        1.231      1.022     1.007     0.120
Size2_BM3        1.200      0.992     0.832     0.343
Size2_BM4        1.211      0.967     0.829     0.587
Size2_BM5        1.373      1.064     0.927     0.885
Size3_BM1        1.246      1.125     0.811    -0.218
Size3_BM2        1.129      1.014     0.565     0.051
Size3_BM3        1.119      0.985     0.452     0.323
Size3_BM4        1.168      0.993     0.480     0.565
Size3_BM5        1.353      1.112     0.589     0.864
Size4_BM1        1.087      1.072     0.341    -0.328
Size4_BM2        1.090      1.028

In [35]:
### PS5 (c) — Cell 5: monthly cross-sectional regressions for (1), (2), (3)
# Spec (1) uses the PS3 market-only beta; specs (2), (3) use the joint FF3 loadings.
specs = {
    "(1)": ["beta_M_capm", "ln_size", "ln_BM"],
    "(2)": ["beta_M_3f", "beta_SMB", "beta_HML"],
    "(3)": ["beta_M_3f", "ln_size", "ln_BM", "beta_SMB", "beta_HML"],
}
labels = {
    "const": "gamma_0",
    "beta_M_capm": "gamma_M",
    "beta_M_3f": "gamma_M",
    "ln_size": "gamma_size",
    "ln_BM": "gamma_BM",
    "beta_SMB": "gamma_SMB",
    "beta_HML": "gamma_HML",
}


def run_cross_sections(exret, betas, ln_size, ln_bm, regressors):
    """Second pass: one OLS per month across the 25 portfolios.
    Betas are the same every month; characteristics change by month."""
    terms = [labels[x] for x in ["const"] + regressors]
    gammas = pd.DataFrame(index=exret.index, columns=terms, dtype=float)
    r2 = pd.Series(index=exret.index, dtype=float)
    nobs = pd.Series(index=exret.index, dtype=float)

    for date in exret.index:
        X_t = betas.copy()
        X_t["ln_size"] = ln_size.loc[date]
        X_t["ln_BM"] = ln_bm.loc[date]
        X_t = sm.add_constant(X_t[regressors])

        fit = sm.OLS(exret.loc[date], X_t).fit()
        gammas.loc[date] = fit.params[["const"] + regressors].to_numpy()
        r2.loc[date] = fit.rsquared
        nobs.loc[date] = fit.nobs

    return gammas, r2, nobs


gammas_all = {}
r2_all = {}
nobs_all = {}

for spec, regressors in specs.items():
    gammas, r2, nobs = run_cross_sections(
        exret_fm, betas, ln_size, ln_bm, regressors
    )
    gammas_all[spec] = gammas
    r2_all[spec] = r2
    nobs_all[spec] = nobs
    print(f"Spec {spec}: {len(gammas)} monthly regressions, "
          f"regressors = {regressors}")

Spec (1): 1194 monthly regressions, regressors = ['beta_M_capm', 'ln_size', 'ln_BM']
Spec (2): 1194 monthly regressions, regressors = ['beta_M_3f', 'beta_SMB', 'beta_HML']
Spec (3): 1194 monthly regressions, regressors = ['beta_M_3f', 'ln_size', 'ln_BM', 'beta_SMB', 'beta_HML']


In [36]:
### PS5 (c) — Cell 6: Fama–MacBeth summary table
def fmb_summary(gammas):
    """Third pass, unchanged from PS3: mean, sd/sqrt(T), t-stat, p-value."""
    T = len(gammas)
    out = pd.DataFrame({
        "Estimate": gammas.mean(),
        "FMB SE": gammas.std(ddof=1) / np.sqrt(T)
    })
    out["t-stat"] = out["Estimate"] / out["FMB SE"]
    out["p-value"] = 2 * stats.t.sf(out["t-stat"].abs(), T - 1)
    return out


term_order = ["gamma_0", "gamma_M", "gamma_size",
              "gamma_BM", "gamma_SMB", "gamma_HML"]
fmb_all = {}
pieces = {}

for spec in specs:
    fmb_all[spec] = fmb_summary(gammas_all[spec])
    pieces[f"Spec {spec}"] = fmb_all[spec].reindex(term_order)

fmb_table = pd.concat(pieces, axis=1)

pd.set_option("display.width", 200)
print(f"First-pass betas: {beta_start} to {beta_end} (T = {beta_T})")
print(f"Second pass:      {exret_fm.index[0]} to {exret_fm.index[-1]} "
      f"(T = {len(exret_fm)}) | N = 25")
print("gamma_M: spec (1) on beta_M_capm (market-only); "
      "specs (2), (3) on beta_M_3f (joint FF3).")
print("Estimates and SEs in percentage points per month; "
      "t-statistics test gamma = 0.\n")
print(fmb_table.round(4).to_string(na_rep=""))

print("\nAverage monthly cross-sectional R-squared:")
for spec in specs:
    print(f"  Spec {spec}: {r2_all[spec].mean():.4f}")

First-pass betas: 1927-01 to 2026-06 (T = 1194)
Second pass:      1927-01 to 2026-06 (T = 1194) | N = 25
gamma_M: spec (1) on beta_M_capm (market-only); specs (2), (3) on beta_M_3f (joint FF3).
Estimates and SEs in percentage points per month; t-statistics test gamma = 0.

           Spec (1)                         Spec (2)                         Spec (3)                        
           Estimate  FMB SE  t-stat p-value Estimate  FMB SE  t-stat p-value Estimate  FMB SE  t-stat p-value
gamma_0      1.7547  0.4228  4.1499  0.0000   1.6433  0.3954  4.1563  0.0000   1.4108  0.3994  3.5325  0.0004
gamma_M     -0.4777  0.3563 -1.3407  0.1803  -0.9042  0.4095 -2.2080  0.0274   0.2049  0.3911  0.5240  0.6004
gamma_size  -0.0674  0.0294 -2.2959  0.0219                                   -0.0932  0.0477 -1.9551  0.0508
gamma_BM     0.2327  0.0570  4.0822  0.0000                                    0.4808  0.1096  4.3887  0.0000
gamma_SMB                                     0.1087  0.0973  1.11

In [37]:
### PS5 (c) — Cell 7: verification checks
# 1. Betas fixed across months; each spec uses the intended beta.
b2 = betas[["beta_M_3f", "beta_SMB", "beta_HML"]]
cs_2 = sm.OLS(exret_fm.mean(), sm.add_constant(b2)).fit()
check1 = np.allclose(cs_2.params.to_numpy(), fmb_all["(2)"]["Estimate"].to_numpy())
X_avg_1 = pd.concat([betas[["beta_M_capm"]], ln_size.mean().rename("ln_size"),
                     ln_bm.mean().rename("ln_BM")], axis=1)
cs_1 = sm.OLS(exret_fm.mean(), sm.add_constant(X_avg_1)).fit()
check1b = not np.allclose(cs_1.params.to_numpy(), fmb_all["(1)"]["Estimate"].to_numpy())
d = exret_fm.index[len(exret_fm) // 2]
check1c = True
for spec, regressors in specs.items():
    X_t = betas.copy()
    X_t["ln_size"] = ln_size.loc[d]
    X_t["ln_BM"] = ln_bm.loc[d]
    X_np = np.column_stack([np.ones(25), X_t[regressors].to_numpy()])
    coef = np.linalg.lstsq(X_np, exret_fm.loc[d].to_numpy(), rcond=None)[0]
    check1c = check1c and np.allclose(coef, gammas_all[spec].loc[d].to_numpy())
print(f"1. Betas estimated once on {beta_start} to {beta_end} (T = {beta_T})")
print("   Spec (2) FMB mean = single OLS on fixed betas:", check1)
print("   Spec (1) FMB mean differs (characteristics vary):", check1b)
print(f"   Hand re-run of {d}: (1) uses beta_M_capm, (2)/(3) use FF3 loadings:", check1c)

# 2. Size is lagged one month (not contemporaneous).
check2 = np.allclose(ln_size.to_numpy(), np.log(size).reindex(exret_fm.index - 1).to_numpy())
check2b = not np.allclose(ln_size.to_numpy(), np.log(size).reindex(exret_fm.index).to_numpy())
print("\n2. ln_size(t) = ln(size(t-1)):", check2, "| not contemporaneous:", check2b)

# 3. BE/ME is the calendar-year Y-1 value for every month of year Y.
check3 = np.allclose(ln_bm.to_numpy(), np.log(bm).reindex(exret_fm.index.year - 1).to_numpy())
check3b = ln_bm.groupby(ln_bm.index.year).nunique().max().max() == 1
check3c = exret_fm.index[0] == pd.Period(f"{bm.index[0] + 1}-01", freq="M")
print("\n3. ln_BM(t) = ln(BE/ME, year(t) - 1):", check3,
      "| one value per calendar year:", check3b,
      f"| sample starts {exret_fm.index[0]}:", check3c)

# 4.-6. Same sample, FMB SE = sd/sqrt(T), 25 portfolios every month.
check4, check5, check6 = True, True, True
for spec in specs:
    g = gammas_all[spec]
    check4 = check4 and g.index.equals(exret_fm.index) and g.notna().all().all()
    se_manual = g.std(ddof=1) / np.sqrt(len(g))
    check5 = check5 and np.allclose(se_manual, fmb_all[spec]["FMB SE"])
    check6 = check6 and (nobs_all[spec] == 25).all()
print("\n4. Same monthly sample in all specs:", check4, f"(T = {len(exret_fm)})")
print("5. FMB SE = std(gamma_t)/sqrt(T):", check5)
print("6. 25 portfolios in every cross section:", check6)
assert check1 and check1b and check1c and check2 and check2b
assert check3 and check3b and check3c and check4 and check5 and check6

# PS3 replication of eq (1): 1969-07 to 2026-06, market-only beta re-estimated
# on that sample, same lags (team rule = PS3 rule).
ps3_start = "1969-07"
exret_ps3 = exret.loc[ps3_start:]
ln_size_ps3 = ln_size_all.loc[ps3_start:]
ln_bm_ps3 = ln_bm_all.loc[ps3_start:]
betas_ps3 = pd.DataFrame(index=names, columns=["beta_M_capm"], dtype=float)
X_market_ps3 = sm.add_constant(rm.loc[exret_ps3.index])
for name in names:
    betas_ps3.loc[name, "beta_M_capm"] = sm.OLS(exret_ps3[name], X_market_ps3).fit().params["MktRF"]
g_ps3, _, _ = run_cross_sections(exret_ps3, betas_ps3, ln_size_ps3, ln_bm_ps3, specs["(1)"])
fmb_ps3 = fmb_summary(g_ps3)

# Printed by PS3 notebook cell ### 1(f) (4 d.p.).
ps3_printed = pd.Series({"gamma_0": 1.1539, "gamma_M": -0.2646,
                         "gamma_size": -0.0194, "gamma_BM": 0.1573})
ps3_t = pd.Series({"gamma_0": 3.4017, "gamma_M": -0.8442,
                   "gamma_size": -0.6428, "gamma_BM": 2.1055})
compare = pd.DataFrame({"PS3 est.": ps3_printed, "Replication est.": fmb_ps3["Estimate"],
                        "PS3 t": ps3_t, "Replication t": fmb_ps3["t-stat"],
                        "PS5 full-sample est.": fmb_all["(1)"]["Estimate"]})
print(f"\nPS3 replication, eq (1): {exret_ps3.index[0]} to {exret_ps3.index[-1]} "
      f"(T = {len(exret_ps3)})")
print(compare.round(4).to_string())
match = (np.allclose(fmb_ps3["Estimate"], ps3_printed, atol=5e-5)
         and np.allclose(fmb_ps3["t-stat"], ps3_t, atol=5e-5))
print("Estimates and t-stats reproduce PS3 to 4 d.p.:", match)
assert match

1. Betas estimated once on 1927-01 to 2026-06 (T = 1194)
   Spec (2) FMB mean = single OLS on fixed betas: True
   Spec (1) FMB mean differs (characteristics vary): True
   Hand re-run of 1976-10: (1) uses beta_M_capm, (2)/(3) use FF3 loadings: True

2. ln_size(t) = ln(size(t-1)): True | not contemporaneous: True

3. ln_BM(t) = ln(BE/ME, year(t) - 1): True | one value per calendar year: True | sample starts 1927-01: True

4. Same monthly sample in all specs: True (T = 1194)
5. FMB SE = std(gamma_t)/sqrt(T): True
6. 25 portfolios in every cross section: True

PS3 replication, eq (1): 1969-07 to 2026-06 (T = 684)
            PS3 est.  Replication est.   PS3 t  Replication t  PS5 full-sample est.
gamma_0       1.1539            1.1539  3.4017         3.4017                1.7547
gamma_M      -0.2646           -0.2646 -0.8442        -0.8442               -0.4777
gamma_size   -0.0194           -0.0194 -0.6428        -0.6428               -0.0674
gamma_BM      0.1573            0.1573  2.105

### Sensitivity check: which market beta to use in equation (1)

**Wording ambiguity.** Step 1 of Question (c) says to estimate $\beta_{iM}$, $\beta_{iSMB}$ and $\beta_{iHML}$ "using full-period OLS regressions on RMRF, SMB, and HML," and the same symbol $\beta_{iM}$ appears in all three equations. This can be read as one joint FF3 regression whose market loading is used everywhere. However, the same step notes that "$\beta_{iM}$ was already estimated in Problem Set 3," and we are asked to verify that equation (1) matches the PS3 $\gamma_{size}$ and $\gamma_{B/M}$. In PS3, $\beta_{iM}$ came from a market-only regression. The assignment does not settle which reading is intended. The team convention uses the market-only beta in (1) and the joint FF3 betas in (2) and (3).

**Main specification.** Equation (1) uses `beta_M_capm` (market-only, the same as PS3). Under the PS3 setup (Cell 7), this reproduces the PS3 estimates and t-statistics to 4 d.p. Using the joint FF3 market loading in that setup does not (Cell 8: `gamma_size` = -0.0016 and `gamma_BM` = 0.1819, versus -0.0194 and 0.1573). Equations (2) and (3) use `beta_M_3f`, `beta_SMB` and `beta_HML` from one joint regression.

**What changes with `beta_M_3f` in equation (1).** The sample (1927-01 to 2026-06, T = 1194), characteristics and FMB SE are the same:

| | `beta_M_capm` (main) | | `beta_M_3f` (sensitivity) | |
|---|---:|---:|---:|---:|
| | Estimate (SE) | t | Estimate (SE) | t |
| `gamma_0` | 1.7547 (0.4228) | 4.15 | 1.5272 (0.4066) | 3.76 |
| `gamma_M` | -0.4777 (0.3563) | -1.34 | -0.5264 (0.4346) | -1.21 |
| `gamma_size` | -0.0674 (0.0294) | -2.30 | -0.0367 (0.0305) | -1.20 |
| `gamma_BM` | 0.2327 (0.0570) | 4.08 | 0.2020 (0.0566) | 3.57 |

Average monthly $R^2$ moves from 0.4898 to 0.4852, and the two market betas have a cross-sectional correlation of 0.65. The main difference is `gamma_size`: its |t| is above 1.96 with `beta_M_capm` but not with `beta_M_3f`. `gamma_BM` has t above 3 in both versions.

## Question (c) Report

**How the PS3 code was extended.** I started from my PS3 cells `### 1(b)` (market-only first pass, monthly cross-sections, FMB SEs) and `### 1(f)` (lagged size and BE/ME). Comparing those cells with this notebook, the visible changes are:

1. Data loading uses `Problem_Set5_2026.xlsx`. Mkt-RF, SMB, HML and RF come from the "Fama-French factors" sheet. PS3's fixed `1969-07` start was replaced by the team's common sample: all months where returns, factors and characteristics exist (1927-01 to 2026-06, T = 1194).
2. First pass, on the same months: the PS3 market-only regression is kept (`beta_M_capm`). A joint regression on RMRF, SMB and HML adds `beta_M_3f`, `beta_SMB` and `beta_HML`.
3. Second pass: PS3's monthly loop became one function, `run_cross_sections()`, driven by a `specs` dictionary, so equations (1)–(3) share the same code, sample and inputs.
4. Third pass: the PS3 averaging code was moved into `fmb_summary()` without changes, and the results are combined into a side-by-side table.
5. Characteristic timing is the same as PS3 and the team convention: size at $t-1$, and BE/ME of calendar year $Y-1$ for every month of year $Y$.

This list describes what can be seen in the code. The code alone does not show which edits came from the LLM and which came from my own revisions.

**Were the changes correct?** Yes. All checks in Cell 7 pass:
- Equation (2) equals a single regression of average returns on the fixed betas.
- A numpy re-run of one month confirms which beta each equation uses.
- All equations use the same 1,194 months with 25 portfolios each.
- Every SE equals $sd(\hat\gamma_t)/\sqrt{T}$.

The team-module check also matches the shared `fmb.py` exactly.

**Fixed betas, time-varying characteristics.** The betas are estimated once and stay fixed across all months of the second pass. Size and BE/ME are lagged and change over time. Checks 1–3 in Cell 7 confirm this structure was preserved.

**Results** (percent per month; t-statistics test $\gamma = 0$; 1927-01 to 2026-06, T = 1194, N = 25):

| Coefficient | Eq (1) Est. | Eq (1) SE | Eq (1) t | Eq (2) Est. | Eq (2) SE | Eq (2) t | Eq (3) Est. | Eq (3) SE | Eq (3) t |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| `gamma_0` | 1.7547 | 0.4228 | 4.15 | 1.6433 | 0.3954 | 4.16 | 1.4108 | 0.3994 | 3.53 |
| `gamma_M` | -0.4777 | 0.3563 | -1.34 | -0.9042 | 0.4095 | -2.21 | 0.2049 | 0.3911 | 0.52 |
| `gamma_size` | -0.0674 | 0.0294 | -2.30 | | | | -0.0932 | 0.0477 | -1.96 |
| `gamma_BM` | 0.2327 | 0.0570 | 4.08 | | | | 0.4808 | 0.1096 | 4.39 |
| `gamma_SMB` | | | | 0.1087 | 0.0973 | 1.12 | -0.1567 | 0.1892 | -0.83 |
| `gamma_HML` | | | | 0.3752 | 0.1061 | 3.54 | -0.4656 | 0.2270 | -2.05 |

Note: $\beta_M$ in equation (1) is the market-only beta, and $\beta_M$ in equations (2) and (3) is the joint FF3 loading. They are different numbers.

**Check against PS3.** On 1969-07 to 2026-06 (T = 684), with the market-only beta re-estimated on that sample, equation (1) gives `gamma_size` = -0.0194 (t = -0.64) and `gamma_BM` = 0.1573 (t = 2.11). These match my PS3 results exactly to 4 d.p., and `gamma_0` and `gamma_M` also match. The full-sample equation (1) estimates differ because the sample period is different.

**Implementation qualifications.**
- *Market beta in equation (1):* the wording is ambiguous (see the sensitivity note). With `beta_M_3f`, `gamma_size` changes from -0.0674 (t = -2.30) to -0.0367 (t = -1.20), while `gamma_BM` stays at t > 3.
- *BE/ME timing:* this follows the PS3/team calendar-year rule, not the Fama–French July–June timing described in Lecture 2.
- *Generated regressors:* the betas are estimated on the full sample, so they contain measurement error and a look-ahead element. The FMB SEs are not adjusted for beta-estimation error.
- *PS3 comparison precision:* the match is confirmed to 4 d.p., which is all PS3 printed.

In [38]:
### (d) — Step 1
key_terms = ["gamma_M", "gamma_size", "gamma_BM", "gamma_SMB", "gamma_HML"]
cols = ["Estimate", "FMB SE", "t-stat"]

pieces_d = {}
for spec in specs:
    pieces_d[f"Eq {spec}"] = fmb_all[spec].reindex(key_terms)[cols]
evidence = pd.concat(pieces_d, axis=1)

print(f"Second pass: {exret_fm.index[0]} to {exret_fm.index[-1]} "
      f"(T = {len(exret_fm)}), N = 25. Units: % per month.")
print("gamma_M: Eq (1) on beta_M_capm; Eq (2), (3) on beta_M_3f.\n")
print(evidence.round(4).to_string(na_rep=""))

# How each coefficient changes when both sets of variables are in Eq (3).
change = pd.DataFrame(index=["gamma_size", "gamma_BM", "gamma_SMB", "gamma_HML"],
                      columns=["Alone in", "Est. alone", "t alone",
                               "Est. in (3)", "t in (3)",
                               "SE alone", "SE in (3)"])
for term in change.index:
    if term in ["gamma_size", "gamma_BM"]:
        alone = "(1)"
    else:
        alone = "(2)"
    change.loc[term] = [
        alone,
        fmb_all[alone].loc[term, "Estimate"], fmb_all[alone].loc[term, "t-stat"],
        fmb_all["(3)"].loc[term, "Estimate"], fmb_all["(3)"].loc[term, "t-stat"],
        fmb_all[alone].loc[term, "FMB SE"], fmb_all["(3)"].loc[term, "FMB SE"],
    ]
print("\nAlone vs. combined (Eq 3):")
print(change.to_string(float_format=lambda v: f"{v:.4f}"))

avg_char = pd.DataFrame({"avg ln_size": ln_size.mean(), "avg ln_BM": ln_bm.mean()})
corr = pd.concat([betas[["beta_SMB", "beta_HML"]], avg_char], axis=1).corr()
print("\nCross-sectional correlations (25 portfolios):")
print(corr.loc[["beta_SMB", "beta_HML"], ["avg ln_size", "avg ln_BM"]].round(3).to_string())

Second pass: 1927-01 to 2026-06 (T = 1194), N = 25. Units: % per month.
gamma_M: Eq (1) on beta_M_capm; Eq (2), (3) on beta_M_3f.

             Eq (1)                   Eq (2)                  Eq (3)                
           Estimate  FMB SE  t-stat Estimate  FMB SE t-stat Estimate  FMB SE  t-stat
gamma_M     -0.4777  0.3563 -1.3407  -0.9042  0.4095 -2.208   0.2049  0.3911  0.5240
gamma_size  -0.0674  0.0294 -2.2959                          -0.0932  0.0477 -1.9551
gamma_BM     0.2327  0.0570  4.0822                           0.4808  0.1096  4.3887
gamma_SMB                             0.1087  0.0973  1.117  -0.1567  0.1892 -0.8283
gamma_HML                             0.3752  0.1061  3.536  -0.4656  0.2270 -2.0511

Alone vs. combined (Eq 3):
           Alone in Est. alone t alone Est. in (3) t in (3) SE alone SE in (3)
gamma_size      (1)    -0.0674 -2.2959     -0.0932  -1.9551   0.0294    0.0477
gamma_BM        (1)     0.2327  4.0822      0.4808   4.3887   0.0570    0.1096
gamma_SMB

## Question (d)

### Step 1

Equation (3), which includes both characteristics and factor betas, fits best, with an average cross-sectional $R^2$ of 0.573, compared with 0.490 for equation (1) and 0.485 for equation (2). In the combined specification, BE/ME stays strongly significant, and its coefficient roughly doubles relative to equation (1) (0.4808, $t = 4.39$, versus 0.2327, $t = 4.08$). Size remains negative but narrowly misses the 5% level ($t = -1.96$, $p = 0.0508$). By contrast, the HML beta premium turns from positive and significant in equation (2) (0.3752, $t = 3.54$) to negative in equation (3) (−0.4656, $t = -2.05$), and SMB is insignificant in both equations ($t = -0.83$ in (3)). Taken together, BE/ME keeps explanatory power after controlling for factor exposures, while the betas lose their expected positive sign, which leans toward a characteristics interpretation; however, the roughly doubled standard errors in equation (3) suggest betas and characteristics are hard to separate, so I treat this conclusion as tentative.

### Step 2 — LLM discussion


**1. "What is the strongest economic argument that size and BE/ME are priced because they are characteristics rather than because they co-vary with systematic risk?"**

The strongest argument is that investors may systematically misprice firms with certain observable characteristics. For example, investors may overvalue glamorous, low-BE/ME firms and undervalue distressed or neglected high-BE/ME firms, so the later return premium reflects price correction rather than compensation for bearing systematic risk. Under this view, size and BE/ME themselves should predict returns even after controlling for factor betas.

**2. "What is the strongest economic argument that size and BE/ME premia are compensation for systematic risk — and what would that imply about the betas on SMB and HML?"**

The risk view says small and high-BE/ME firms tend to perform poorly in certain bad economic states, so investors require higher expected returns to hold them. SMB and HML are then factor-mimicking portfolios capturing these common systematic risks. Therefore, firms with higher \(\beta_{SMB}\) or \(\beta_{HML}\) should earn higher average returns, and once those betas are controlled for, size and BE/ME should have little additional explanatory power.

**3. "In a Fama–MacBeth horse race between equations (1), (2), and (3), what empirical pattern would cleanly support the characteristics view? What pattern would support the risk view?"**

Suppose equation (1) contains characteristics, equation (2) contains factor betas, and equation (3) includes both:
- Characteristics view: size and BE/ME are significant in (1) and remain significant in (3), while the SMB/HML beta prices become weak or insignificant.
- Risk view: SMB/HML betas are significant in (2) and remain significant in (3), while the size and BE/ME coefficients become weak or insignificant.
- If both remain significant in (3), the evidence is mixed rather than a clean victory for either explanation.

### Evaluation： 


1. Does the LLM correctly identify the key distinguishing prediction — that if characteristics dominate in equation (3), it is hard to sustain a pure risk story?

The LLM correctly identifies that characteristics remaining significant in equation (3), while beta premia weaken, would challenge a pure SMB/HML risk explanation. However, significance alone is not enough; the signs of the coefficients also matter.

2. Does the LLM correctly note that the horse race is inherently noisy because characteristics and betas are correlated by construction (size sorts produce size beta differences)?

The revised answer does not mention that characteristics and betas are strongly correlated because of the portfolio sorts. My notebook shows correlations of −0.974 between average log size and SMB beta and 0.948 between average log BE/ME and HML beta, making their separate effects difficult to identify.

3. Does your empirical evidence from equations (1)–(3) more strongly support one side? Be explicit about which coefficients drive your conclusion.
 
My full-sample results lean toward the characteristics view, especially for BE/ME: its coefficient increases from 0.2327 (\(t=4.08\)) in equation (1) to 0.4808 (\(t=4.39\)) in equation (3). Size remains negative but narrowly misses 5% significance (\(p=0.0508\)), while SMB is insignificant in both equations (2) and (3). HML remains significant but reverses from positive to negative (−0.4656, \(t=-2.05\)), which conflicts with the positive premium predicted by the LLM’s risk explanation. The evidence therefore favors characteristics, but the strong correlations make this conclusion tentative

In [39]:
### PS5 (e): repeat (c) using January 1963 onward (team conventions)
# Same lags as (c); returns from 1963-01; betas re-estimated on this sample.
months_e = fm_months[fm_months >= pd.Period("1963-01", freq="M")]
exret_e = exret.loc[months_e]

betas_e = pd.DataFrame(index=names, columns=betas.columns, dtype=float)
X_m = sm.add_constant(rm.loc[months_e])
X_f = sm.add_constant(f3.loc[months_e])
for name in names:
    betas_e.loc[name, "beta_M_capm"] = sm.OLS(exret_e[name], X_m).fit().params["MktRF"]
    betas_e.loc[name, ["beta_M_3f", "beta_SMB", "beta_HML"]] = (
        sm.OLS(exret_e[name], X_f).fit().params[["MktRF", "SMB", "HML"]].to_numpy())

fmb_e, r2_e, pieces_e = {}, {}, {}
for spec, regressors in specs.items():
    g, r2, nobs = run_cross_sections(exret_e, betas_e, ln_size_all.loc[months_e],
                                     ln_bm_all.loc[months_e], regressors)
    assert g.notna().all().all() and (nobs == 25).all()
    fmb_e[spec] = fmb_summary(g)
    r2_e[spec] = r2.mean()
    pieces_e[f"Spec {spec}"] = fmb_e[spec].reindex(term_order)

assert months_e[0] == pd.Period("1963-01", freq="M")
assert months_e.equals(pd.period_range(months_e[0], months_e[-1], freq="M"))

table_e = pd.concat(pieces_e, axis=1)
print(f"Sample: {months_e[0]} to {months_e[-1]} | T = {len(months_e)} | N = 25 "
      "| betas estimated on the same months")
print("gamma_M: spec (1) on beta_M_capm; specs (2), (3) on beta_M_3f "
      "(different numbers). % per month.\n")
print(table_e.round(4).to_string(na_rep=""))
for spec in specs:
    print(f"Average monthly R-squared, spec {spec}: {r2_e[spec]:.4f}")

Sample: 1963-01 to 2026-06 | T = 762 | N = 25 | betas estimated on the same months
gamma_M: spec (1) on beta_M_capm; specs (2), (3) on beta_M_3f (different numbers). % per month.

           Spec (1)                         Spec (2)                         Spec (3)                        
           Estimate  FMB SE  t-stat p-value Estimate  FMB SE  t-stat p-value Estimate  FMB SE  t-stat p-value
gamma_0      1.2790  0.3223  3.9678  0.0001   1.1429  0.2583  4.4247  0.0000   1.3700  0.3828  3.5791  0.0004
gamma_M     -0.2247  0.2901 -0.7746  0.4388  -0.5261  0.3035 -1.7337  0.0834  -0.2827  0.3631 -0.7787  0.4364
gamma_size  -0.0437  0.0287 -1.5202  0.1289                                    0.0045  0.0507  0.0892  0.9289
gamma_BM     0.1704  0.0682  2.5007  0.0126                                    0.4876  0.1368  3.5652  0.0004
gamma_SMB                                     0.1228  0.1121  1.0951  0.2738   0.0527  0.2055  0.2564  0.7977
gamma_HML                                     0.33

## Question (e)

Post-1963 sample: 1963-01 to 2026-06 (T = 762), with betas re-estimated on the same months (team conventions). Full sample: 1927-01 to 2026-06 (T = 1194).

| Variable | Eq | Full: Est. (t) | Post-1963: Est. (t) |
|---|---|---:|---:|
| $\gamma_{size}$ | (1) | -0.0674 (-2.30) | -0.0437 (-1.52) |
| $\gamma_{B/M}$ | (1) | 0.2327 (4.08) | 0.1704 (2.50) |
| $\gamma_{SMB}$ | (2) | 0.1087 (1.12) | 0.1228 (1.10) |
| $\gamma_{HML}$ | (2) | 0.3752 (3.54) | 0.3364 (3.06) |
| $\gamma_{size}$ | (3) | -0.0932 (-1.96) | 0.0045 (0.09) |
| $\gamma_{B/M}$ | (3) | 0.4808 (4.39) | 0.4876 (3.57) |
| $\gamma_{SMB}$ | (3) | -0.1567 (-0.83) | 0.0527 (0.26) |
| $\gamma_{HML}$ | (3) | -0.4656 (-2.05) | -0.4949 (-1.87) |

**Does the result change?** For BE/ME, the result is basically the same. In equation (3), $\gamma_{B/M}$ is almost unchanged (0.4876, t = 3.57), while the HML beta premium stays negative and is now insignificant at 5% (t = -1.87, p = 0.06). So the value evidence still leans toward the characteristics view. The size result changes: $\gamma_{size}$ is no longer significant in equation (1) (t = -1.52) and is essentially zero in equation (3) (t = 0.09), and SMB is insignificant in both samples. After 1963, neither size nor its beta explains average returns, so the size dimension gives no evidence for either side.

**More or less confident?** I am more confident about BE/ME, because its coefficient in equation (3) has a similar size and stays significant in both samples. I am less confident about size, which was already borderline in part (d) and disappears after 1963. Overall the conclusion is still tentative. With fewer months, the standard errors in equation (3) are larger (e.g. $\gamma_{HML}$ SE 0.2644 vs 0.2270), and betas and characteristics are still strongly correlated by construction of the 25 portfolios.

**Why might the sample period matter?**
- *Data quality:* before 1963, book equity is not from Compustat and covers fewer firms, so BE/ME and the portfolios may be measured less precisely.
- *In-sample vs. out-of-sample:* the post-1963 period is close to the sample in which the size and value effects were originally documented, so the pre-1963 data works partly as an out-of-sample check.
- *Changing premia and risk:* the size premium seems to depend on the period. The pre-1963 sample also includes the Great Depression, when distress risk was especially important. If factor loadings change over time, full-sample fixed betas fit some periods worse than others.